# Module 3: Data Visualization

**BigData Analytics Course — AKTN Magister Program**

---

## Learning Objectives
1. Create publication-quality charts with **Matplotlib** and **Seaborn**.
2. Build interactive dashboards with **Plotly**.
3. Choose the right chart type for the right analytical question.
4. Visualise distributions, correlations, time series, and geospatial data.
5. Apply design principles (colour, annotation, layout) to communicate insights.

**Estimated time:** 120 minutes  
**Prerequisites:** Module 2

In [ ]:
# Install Plotly if not already present (pre-installed on Colab)
# !pip install plotly --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Seaborn styling
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})

print("✅ Libraries loaded.")

In [ ]:
# ── Recreate the e-commerce dataset from Module 2 ───────────────────────────
rng = np.random.default_rng(42)
n = 2_000
categories = ['Electronics', 'Clothing', 'Books', 'Home & Garden', 'Sports']
regions    = ['North', 'South', 'East', 'West']

df = pd.DataFrame({
    'date'        : pd.date_range('2024-01-01', periods=n, freq='h'),
    'category'    : rng.choice(categories, n),
    'region'      : rng.choice(regions, n),
    'quantity'    : rng.integers(1, 20, n),
    'unit_price'  : rng.uniform(5, 500, n).round(2),
    'discount_pct': rng.choice([0, 5, 10, 15, 20], n),
    'customer_age': rng.integers(18, 70, n),
})
df['revenue'] = (df['quantity'] * df['unit_price'] * (1 - df['discount_pct'] / 100)).round(2)
df['month']   = df['date'].dt.month_name()
df['weekday'] = df['date'].dt.day_name()

print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

## 1. Matplotlib — Foundation of Python Visualisation

Matplotlib gives you full control. We use the **object-oriented API** (`fig, ax = plt.subplots()`) for reproducible, customisable charts.

In [ ]:
# ── 1.1 Distribution of revenue — histogram + KDE ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['revenue'], bins=60, color='#2196F3',
             edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Revenue ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Revenue Distribution (Histogram)')
axes[0].axvline(df['revenue'].mean(),   color='red',   ls='--', label='Mean')
axes[0].axvline(df['revenue'].median(), color='orange', ls='--', label='Median')
axes[0].legend()

# Box-plot by category
categories_ordered = df.groupby('category')['revenue'].median().sort_values().index
data_per_cat = [df[df['category'] == c]['revenue'].values for c in categories_ordered]
bp = axes[1].boxplot(data_per_cat, labels=categories_ordered,
                     patch_artist=True, notch=True)
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Revenue ($)')
axes[1].set_title('Revenue by Category (Box Plot)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# ── 1.2 Heatmap of weekly / hourly revenue pattern ──────────────────────────
df['hour'] = df['date'].dt.hour

heatmap_data = (
    df.groupby(['weekday', 'hour'])['revenue']
    .mean()
    .unstack(fill_value=0)
)

# Re-order days of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex([d for d in day_order if d in heatmap_data.index])

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(heatmap_data.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(24))
ax.set_xticklabels(range(24), fontsize=8)
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index)
ax.set_xlabel('Hour of Day')
ax.set_title('Average Revenue — Day of Week × Hour of Day')
plt.colorbar(im, ax=ax, label='Avg Revenue ($)')
plt.tight_layout()
plt.show()

## 2. Seaborn — Statistical Visualisation

In [ ]:
# ── 2.1 Pairplot — multivariate relationships ───────────────────────────────
sample = df.sample(500, random_state=42)
pair_vars = ['revenue', 'quantity', 'unit_price', 'customer_age']

g = sns.pairplot(sample[pair_vars + ['category']],
                 hue='category', diag_kind='kde',
                 plot_kws=dict(alpha=0.5, s=20))
g.fig.suptitle('Pairplot — Numeric Variables Coloured by Category',
               y=1.01, fontsize=14)
plt.show()

In [ ]:
# ── 2.2 Correlation heatmap ─────────────────────────────────────────────────
num_cols = ['revenue', 'quantity', 'unit_price', 'discount_pct', 'customer_age']
corr = df[num_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5, ax=ax
)
ax.set_title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.3 Violin + strip plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
sns.violinplot(data=df, x='category', y='revenue', hue='category',
               inner=None, alpha=0.5, ax=ax, legend=False)
sns.stripplot(data=df.sample(400, random_state=1), x='category', y='revenue',
              color='black', alpha=0.2, size=2, ax=ax)
ax.set_title('Revenue Distribution by Category (Violin + Strip)', fontsize=13)
ax.set_xlabel('Category')
ax.set_ylabel('Revenue ($)')
plt.tight_layout()
plt.show()

## 3. Plotly — Interactive Visualisation

Plotly charts are **fully interactive** in Colab and Jupyter — hover, zoom, and filter in the browser.

In [ ]:
# ── 3.1 Interactive bar chart — revenue by region and category ───────────────
rev_rc = (
    df.groupby(['region', 'category'])['revenue']
    .sum()
    .reset_index()
)

fig = px.bar(
    rev_rc, x='region', y='revenue', color='category',
    barmode='group',
    title='Total Revenue by Region and Category',
    labels={'revenue': 'Revenue ($)', 'region': 'Region'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(legend_title_text='Category')
fig.show()

In [ ]:
# ── 3.2 Sunburst chart — hierarchical revenue breakdown ─────────────────────
fig = px.sunburst(
    rev_rc, path=['region', 'category'], values='revenue',
    title='Revenue Hierarchy: Region → Category',
    color='revenue',
    color_continuous_scale='Blues'
)
fig.show()

In [ ]:
# ── 3.3 Animated scatter — daily revenue over time ──────────────────────────
daily = (
    df.assign(day=df['date'].dt.date)
    .groupby(['day', 'category'])
    .agg(revenue=('revenue', 'sum'), orders=('quantity', 'count'))
    .reset_index()
)
daily['day'] = daily['day'].astype(str)

fig = px.scatter(
    daily.head(1000), x='orders', y='revenue',
    color='category', size='revenue',
    hover_name='category',
    title='Daily Orders vs Revenue (by Category)',
    labels={'orders': 'Number of Orders', 'revenue': 'Daily Revenue ($)'},
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Plotly
)
fig.show()

## 4. Visualisation Best Practices

| Principle | Do ✅ | Don't ❌ |
|-----------|-------|----------|
| **Chart type** | Match chart to data type and question | Use pie chart for > 5 categories |
| **Colour** | Use colour-blind-friendly palettes | Use red/green as only distinction |
| **Annotation** | Label key data points | Clutter every point |
| **Axes** | Always label axes with units | Start y-axis at non-zero (usually) |
| **Title** | Describe the insight, not just the chart | "Chart 1" as a title |
| **Data-ink ratio** | Remove chart-junk (excess gridlines, borders) | Heavy borders, 3-D effects |

### Choosing the Right Chart

| Question | Recommended Chart |
|----------|------------------|
| Distribution of one variable | Histogram, KDE, Box plot |
| Comparison across categories | Bar chart, Grouped bar |
| Relationship between two numerics | Scatter plot |
| Trend over time | Line chart |
| Correlation matrix | Heatmap |
| Part-to-whole | Stacked bar, Treemap, Sunburst |
| Geographic data | Choropleth map |

In [ ]:
# ── 4.1 Dashboard — combining multiple charts ───────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Revenue by Category',
        'Orders by Region',
        'Price vs Revenue',
        'Age Distribution',
    ]
)

# Revenue by category (bar)
rc = df.groupby('category')['revenue'].sum().reset_index()
fig.add_trace(go.Bar(x=rc['category'], y=rc['revenue'],
                     marker_color='#2196F3', name='Revenue'), row=1, col=1)

# Orders by region (pie)
ro = df.groupby('region').size().reset_index(name='orders')
fig.add_trace(go.Pie(labels=ro['region'], values=ro['orders'],
                     name='Orders'), row=1, col=2)

# Price vs revenue (scatter)
samp = df.sample(300, random_state=42)
fig.add_trace(go.Scatter(x=samp['unit_price'], y=samp['revenue'],
                         mode='markers', marker=dict(opacity=0.5, size=4),
                         name='Price/Revenue'), row=2, col=1)

# Age histogram
fig.add_trace(go.Histogram(x=df['customer_age'], nbinsx=20,
                           marker_color='#4CAF50', name='Age'), row=2, col=2)

fig.update_layout(height=650, title_text='E-Commerce Analytics Dashboard',
                  showlegend=False)
fig.show()

## 5. Summary

| Library | Strengths |
|---------|-----------|
| **Matplotlib** | Full control, publication-quality static charts |
| **Seaborn** | Statistical plots with elegant defaults, built on Matplotlib |
| **Plotly** | Interactive, web-ready charts; dashboards with Dash |

## 📝 Exercises

1. **Matplotlib:** Create a grouped bar chart showing the average discount percentage per category, with error bars representing the standard deviation.
2. **Seaborn:** Use `sns.FacetGrid` to plot a KDE of revenue for each region side-by-side.
3. **Plotly:** Build an animated bar chart (using `px.bar` with `animation_frame`) showing monthly revenue per category.

---
⬅️ [Module 2 — Data Processing with Python](Module_02_Data_Processing_with_Python.ipynb)  
➡️ [Module 4 — Machine Learning Fundamentals](Module_04_Machine_Learning_Fundamentals.ipynb)